# 0. Imports

In [ ]:
%%capture
%pip install easyocr pytesseract codecarbon

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, AutoModelForImageTextToText, LightOnOcrForConditionalGeneration, LightOnOcrProcessor
import numpy as np
import matplotlib.pyplot as plt
import cv2
import datetime as dt
import os, json, re, time
from bs4 import BeautifulSoup
import easyocr
import pytesseract
from codecarbon import EmissionsTracker

# 1. Parameters

In [ ]:
avail_models = ["easyocr", "pytesseract", "google/gemma-3-4b-it", "lightonai/LightOnOCR-2-1B", "zai-org/GLM-OCR", "baidu/Unlimited-OCR", "PaddlePaddle/PaddleOCR-VL-1.6", "Qwen/Qwen2.5-VL-7B-Instruct"]
avail_datasets = ["ICDAR03/apanar", "ICDAR03/lfsosa", "ICDAR03/ryoungt1", "ICDAR03/ryoungt2", "ICDAR13/train", "ICDAR13/test", "ICDAR15/train", "ICDAR15/test", "kahua-ml/flattened-nameplate", "kahua-ml/nameplates-v2", "SVT/train", "icare/train"]

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "cpu" else torch.bfloat16

BATCH_SIZE = 50 if device == "cuda" else 10
HF_TOKEN = os.environ.get("HF_TOKEN", None)
MARGIN = 10 # pixels to add to bounding boxes when cropping
NBR_TOKENS = 1024
USE_BBOX = True
IS_LLM = None
CONF_THRESHOLD = 65

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    is_colab = True
except ImportError:
    print("Google Colab environment not detected.")
    is_colab = False

If we are on Colab, we need to update pytesseract to a version >= 5.0

PS: It asks you to press 'ENTER' to continue

In [ ]:
if is_colab:
    !sudo add-apt-repository ppa:alex-p/tesseract-ocr-devel
    !sudo apt-get update
    !sudo apt-get install tesseract-ocr
    # Get the language packages from version 4
    os.environ["TESSDATA_PREFIX"] = "/usr/share/tesseract-ocr/4.00/tessdata"
    print(pytesseract.get_tesseract_version())

In [ ]:
model_name:str = "lightonai/LightOnOCR-2-1B"
datasets:list[str] = avail_datasets

DATASET_PATH = "resources/datasets/" if not is_colab else "/content/drive/MyDrive/Thesis/resources/datasets/"
SAVE_PATH = "output/predictions/" if not is_colab else "/content/drive/MyDrive/Thesis/resources/predictions/"

# 2. Model preparation
Model download and initialization

In [ ]:
class TesseractOCR:
    """
    A wrapper class for Tesseract OCR using pytesseract.
    It allows for easy configuration of language, page segmentation mode (PSM).
    """
    def __init__(self, lang="eng", psm=6):
        """
        Initialize the TesseractOCR instance.

        Args:
            lang (str): The language to use for OCR. Default is "eng" (English).
            psm (int): The page segmentation mode. Default is 6 (treat the image as a single uniform block of text). If USE_BBOX is True, psm will be set to 7 (assume a single line of text).
        """
        self.lang = lang
        self.psm = psm if USE_BBOX else 6
        self._config = self._build_config()

    def _build_config(self):
        config = f"--psm {self.psm}"
        return config

    def read(self, image):
        return pytesseract.image_to_string(image, lang=self.lang, config=self._config)

    def read_data(self, image):
        return pytesseract.image_to_data(image, lang=self.lang, config=self._config, output_type=pytesseract.Output.DATAFRAME)

class Prediction:
    """
    %%Not Used
    A class to represent a prediction made by an OCR model.
    It contains the predicted word, its confidence score, and the processing time taken to make the prediction.
    """
    def __init__(self, word: str, confidence: float, processing_time: float):
        self.word = word
        self.confidence = confidence

    def __repr__(self):
        return f"Prediction(word={self.word}, confidence={self.confidence}, processing_time={self.processing_time})"

In [ ]:
def init_model(model_name:str):
    global IS_LLM
    IS_LLM = model_name not in ["easyocr", "pytesseract"]
    if IS_LLM:
        global USE_BBOX
        USE_BBOX = False
        model, processor = try_load_llm_model(model_name)
        reader = None
    elif model_name == "easyocr":
        USE_BBOX = True
        reader = easyocr.Reader(['en', 'fr'], gpu=torch.cuda.is_available())
        model, processor = None, None
    elif model_name == "pytesseract":
        USE_BBOX = True
        reader = TesseractOCR(lang="eng", psm=8)
        model, processor = None, None
    else:
        raise ValueError(f"Model {model_name} not supported. Available models: {avail_models}")
    return model, processor, reader

def set_instructions(model_name:str):
    global IS_LLM
    if not IS_LLM:
        return []
    match model_name:
        case "LightOnOCR-2-1B":
            return []
        case _:
            instructions = [
                "You are an OCR engine. You do not chat, explain, or comment.",
                "Locate all text in the image and transcribe it exactly as it appears.",
                "Output ONLY the raw transcribed text. Nothing else.",
                "Do NOT add any introduction (e.g. 'Here is the text...').",
                "Do NOT add any conclusion, offer, or question (e.g. 'Let me know if...').",
                "Do NOT use Markdown formatting: no bullet points, no bold (**), no headers (#), no asterisks.",
                "Do NOT interpret, label, or reformat the content — just transcribe what is visually present, line by line.",
                "Your entire response must be the transcription and nothing else.",
            ]
            return instructions

def try_load_llm_model(model_name):
    model_suffix = model_name.split("/")[-1]
    if os.path.exists(f"Models/{model_suffix}"):
        print(f"Loading model {model_name} from Models/{model_suffix}")
        if model_name == "lightonai/LightOnOCR-2-1B":
            model = LightOnOcrForConditionalGeneration.from_pretrained("lightonai/LightOnOCR-2-1B", dtype=dtype).to(device)
            processor = LightOnOcrProcessor.from_pretrained("lightonai/LightOnOCR-2-1B")
        elif model_name == "PaddlePaddle/PaddleOCR-VL-1.6":
            processor = AutoProcessor.from_pretrained(f"Models/{model_suffix}", use_auth_token=HF_TOKEN)
            model = AutoModelForImageTextToText.from_pretrained(f"Models/{model_suffix}").to(device)
        else:
            processor = AutoProcessor.from_pretrained(f"Models/{model_suffix}", use_auth_token=HF_TOKEN)
            model = AutoModelForMultimodalLM.from_pretrained(f"Models/{model_suffix}").to(device)
        return model, processor
    else:
        print(f"Downloading model {model_name}, and saving in Models/{model_suffix}")
        if model_name == "lightonai/LightOnOCR-2-1B":
            model = LightOnOcrForConditionalGeneration.from_pretrained("lightonai/LightOnOCR-2-1B", dtype=dtype).to(device)
            processor = LightOnOcrProcessor.from_pretrained("lightonai/LightOnOCR-2-1B")
        elif model_name == "PaddlePaddle/PaddleOCR-VL-1.6":
            processor = AutoProcessor.from_pretrained(model_name, use_auth_token=HF_TOKEN)
            model = AutoModelForImageTextToText.from_pretrained(model_name).to(device)
        else:
            processor = AutoProcessor.from_pretrained(model_name, use_auth_token=HF_TOKEN, trust_remote_code=True)
            model = AutoModelForMultimodalLM.from_pretrained(model_name, trust_remote_code=True).to(device)
        processor.save_pretrained(f"Models/{model_suffix}")
        model.save_pretrained(f"Models/{model_suffix}")
        return model, processor

# 3. Inference

## 3.1 Definitions

In [ ]:
def natural_sort_key(value):
    parts = re.split(r'(\d+)', value)
    return [int(part) if part.isdigit() else part.lower() for part in parts]

def save_predictions_checkpoint(path, data):
    # Make sure the directory exists
    os.makedirs(os.path.dirname(path), exist_ok=True)
    temp_path = f"{path}.tmp"
    with open(temp_path, "w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, ensure_ascii=False)
    os.replace(temp_path, path)

def try_resume_from_checkpoint(checkpoint_path):
    if os.path.exists(checkpoint_path) and os.path.getsize(checkpoint_path) > 0:
        with open(checkpoint_path, "r", encoding="utf-8") as handle:
            try:
                data = json.load(handle)
                print(f"Resuming from checkpoint: {checkpoint_path} with {len(data)} entries")
                return data
            except json.JSONDecodeError:
                print(f"Checkpoint file {checkpoint_path} is corrupted. Starting fresh.")
                return {}
    return {}

def show(img, title=""):
    plt.figure(figsize=(8,4))
    if len(img.shape)==2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

def get_resized_roi(image, coords):
    """Get a resized region of interest from the image based on coordinates."""
    if coords is None:
        return image

    if len(coords) == 0:
        return image
    elif len(coords) == 4:
        x1, y1, x2, y2 = map(int, coords)
        x1 = max(0, x1 - MARGIN)
        y1 = max(0, y1 - MARGIN)
        x2 = min(image.shape[1], x2 + MARGIN)
        y2 = min(image.shape[0], y2 + MARGIN)
        roi = image[y1:y2, x1:x2]
    elif len(coords) == 8:
        x1, y1, x2, y2, x3, y3, x4, y4 = map(int, coords)
        x1 = max(0, x1 - MARGIN)
        y1 = max(0, y1 - MARGIN)
        x3 = min(image.shape[1], x3 + MARGIN)
        y3 = min(image.shape[0], y3 + MARGIN)
        roi = image[y1:y3, x1:x3]
    elif len(coords) > 8 and len(coords) % 2 == 0:
        x_s = [coords[i] for i in range(0, len(coords), 2)]
        y_s = [coords[i] for i in range(1, len(coords), 2)]
        x1, y1 = max(0, min(x_s) - MARGIN), max(0, min(y_s) - MARGIN)
        x2, y2 = min(image.shape[1], max(x_s) + MARGIN), min(image.shape[0], max(y_s) + MARGIN)
        roi = image[y1:y2, x1:x2]
    else:
        return None
    return roi

def extract_content(string, model_name:str):
    """Short function to extract the content from the string based on the model name."""
    suffix = model_name.split("/")[-1]
    match suffix:
        case "LightOnOCR-2-1B":
            results = light_on_ocr_to_word_list(string)
            return results
        case "gemma-3-4b-it" | "PaddleOCR-VL-1.6" | "Qwen2.5-VL-7B-Instruct":
            # For gemma-3-4b-it, we need to extract the JSON content
            return gemma_3_to_word_list(string)
        case "NuExtract3":
            return [string]
        case _:
            return [string]

def gemma_3_to_word_list(text: str) -> list[str]:
    words = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue  # on ignore les lignes vides
        words.extend(line.split())  # découpe la ligne en mots sur les espaces
    return words

def light_on_ocr_to_word_list(text: str) -> list[str]:
    # We remove all the text that is after the "Note:" line, as it is not part of the transcription
    img_index = text.lower().find("![image]")
    if img_index != -1:
        text = text[:img_index]
    note_index = text.lower().find("Note:")
    if note_index != -1:
        text = text[:note_index]

    soup = BeautifulSoup(text, "html.parser")
    text_no_html = soup.get_text(separator=" ")

    text_clean = re.sub(r'\|', ' ', text_no_html)
    text_clean = re.sub(r'\$.*?\$', '', text_clean)  # Remove math mode
    text_clean = re.sub(r'(?im)^\s*Note:\s*.*$', '', text_clean)
    text_clean = re.sub(r'^[-:=]{3,}$', '', text_clean, flags=re.MULTILINE)
    text_clean = re.sub(r'#{1,6}\s*', '', text_clean)
    text_clean = re.sub(r'\*{1,2}([^*]+)\*{1,2}', r'\1', text_clean)
    text_clean = re.sub(r'^\s*[-*+]\s+', '', text_clean, flags=re.MULTILINE)

    words = text_clean.strip().split()

    return words

def extract_content_from_bbox_results(results, model_name:str):
    """
    Extract the content from the results based on the model name.
    For easyocr, results is a list of tuples (bbox, text, prob).
    For pytesseract, results is a dictionary with keys 'text' and 'conf'."""
    if model_name == "easyocr":
        conf = 0.0
        word = ""
        for (bbox, text, prob) in results:
            if prob > conf:
                conf = float(prob)
                word = str(text).strip()
        return word, round(conf, 4)
    else:
        # For pytesseract
        conf = 0.0
        word = ""
        for i in range(len(results['text'])):
            text = results['text'][i]
            prob = results['conf'][i]
            if prob > conf:
                conf = float(prob)
                word = str(text).strip()
        return word, round(conf, 4)

def easyocr_pytesseract_to_word_list(results):
    words = []
    if model_name == "easyocr":
        for (bbox, text, prob) in results:
            conf = round(float(prob)*100, 4)
            text = str(text).strip()
            if conf > CONF_THRESHOLD:
                words.append((text, conf))
    else:
        # For pytesseract
        for i in range(len(results['text'])):
            text = results['text'][i]
            prob = results['conf'][i]
            conf = round(float(prob), 4)
            text = str(text).strip()
            if conf > CONF_THRESHOLD:
                words.append((text, conf))
    return words

def save_predictions_to_json(predictions_dict, dataset:str, model_name:str, use_bbox:bool):
    """Save the predictions dictionary to a JSON file."""
    mainset_name, subset_name = dataset.split("/")
    company_name, model_nme = model_name.split("/") if '/' in model_name else (None, model_name)
    # Make sure the directory exists
    save_dir = os.path.join(SAVE_PATH, mainset_name)
    os.makedirs(save_dir, exist_ok=True)
    save_file_path = os.path.join(save_dir, f"{subset_name}_{model_nme}{'_bb' if use_bbox else ''}.json")
    with open(save_file_path, "w", encoding="utf-8") as f:
        json.dump(predictions_dict, f, indent=2, ensure_ascii=False)
    print(f"Predictions saved to {save_file_path}")


In [ ]:
def benchmark(model_name:str = "easyocr", datasets:list[str] = ["ICDAR03/apanar"], use_bbox:bool = True, batch_size:int = BATCH_SIZE, nbr_tokens:int = NBR_TOKENS, use_checkpoint:bool = True):
    # Model initialization
    model, processor, reader = init_model(model_name)
    instructions = set_instructions(model_name)

    # Dataset processing
    for dataset in datasets:
        # Check if dataset exists
        if not os.path.exists(f"{DATASET_PATH}/{dataset}"):
            print(f"Dataset {dataset} not found in {DATASET_PATH}. Skipping.")
            continue

        print(f"Processing dataset {dataset}")
        # The predictions dictionary will store the results for each image in the dataset. The key is the image name (without extension), and the value is a dictionnary with two keys: "predictions" and "time". "predictions" is a list of tuples (word, confidence) and "time" is the processing time in seconds for that image.
        predictions_dict = {}
        batch_dict = {}

        try:
            company_name, model_nme = model_name.split("/")
        except:
            model_nme = model_name
        main, sub = dataset.split("/")

        # Defining the different paths.
        if not is_colab:
            save_path = f"{SAVE_PATH}{main}/{sub}_{model_nme}{'_bb' if use_bbox else ''}.json"
            checkpoint_path = f"resources/checkpoints/{dataset.replace('/', '_')}_{model_nme}{'_bb' if use_bbox else ''}.json"
            emissions_dir = "resources/emissions"
        else:
            save_path = f"/content/drive/MyDrive/Thesis/resources/predictions/{main}/{sub}_{model_nme}{'_bb' if use_bbox else ''}.json"
            checkpoint_path = f"/content/drive/MyDrive/Thesis/resources/checkpoints/{dataset.replace('/', '_')}_{model_nme}{'_bb' if use_bbox else ''}.json"
            emissions_dir = "/content/drive/MyDrive/Thesis/resources/emissions"

        # Load metadata if bounding boxes are used
        if use_bbox:
            metadata_file = json.load(open(f"{DATASET_PATH}{dataset}/data.json"))
            word_count = sum(len(coords_list) for coords_list in metadata_file.values())
            print(f"Metadata loaded for {word_count} words in {len(metadata_file)} images.")

        # Load checkpoint if available
        if use_checkpoint:
            try:
                company_name, model_nme = model_name.split("/")
            except:
                model_nme = model_name

            predictions_dict = try_resume_from_checkpoint(checkpoint_path)

            # If a final prediction file already exists, skip the dataset entirely
            main, sub = dataset.split("/")

            if os.path.exists(save_path):
                print(f"Final predictions already exist at {save_path}. Skipping dataset {dataset}.")
                continue

        # Load all images in the dataset
        image_files = sorted(
            [file for file in os.listdir(f"{DATASET_PATH}/{dataset}") if file.lower().endswith((".jpg", ".jpeg", ".png"))],
            key=lambda file: natural_sort_key(os.path.splitext(file)[0])
        )
        if len(image_files) == 0:
            print(f"No images found in dataset {dataset}. Skipping.")
            continue

        # Start the CodeCarbon tracker
        os.makedirs(emissions_dir, exist_ok=True)
        tracker = EmissionsTracker(project_name=f"{dataset}_{model_name}{'_bb' if use_bbox else ''}", output_dir=emissions_dir, save_to_file=True, log_level="error")
        tracker.start()

        start_time = dt.datetime.now()
        last_checkpoint_time = start_time

        # Process each batch of images
        for batch_start in range(0, len(image_files), batch_size):
            batch_files = image_files[batch_start:batch_start + batch_size]
            batch_names = [os.path.splitext(file)[0] for file in batch_files]

            if not batch_names:
                continue    # Skipping empty batches

            # Treat each image in the batch
            for image_file in batch_files:
                image_path = f"{DATASET_PATH}/{dataset}/{image_file}"
                image_key = os.path.splitext(image_file)[0]
                image = cv2.imread(image_path)

                if image_key in predictions_dict:
                    continue

                if image is None:
                    print(f"Failed to load image {image_path}. Skipping.")
                    continue

                # Processing logic based on model type
                if IS_LLM:
                    # Prepare the input for the LLM model
                    messages = [{"role": "user","content": [{"type": "image", "image": image},{"type": "text", "text": "".join(instructions)},]}]

                    # Start the execution timer for LLM processing
                    exec_start_time = dt.datetime.now() # Mostly to track individual image processing time, to make cactus plot.

                    inputs = processor.apply_chat_template(
                        messages,
                        add_generation_prompt=True,
                        tokenize=True,
                        return_dict=True,
                        return_tensors="pt",
                    ).to(model.device)

                    outputs = model.generate(**inputs, max_new_tokens=nbr_tokens)
                    generated_ids = outputs[0, inputs["input_ids"].shape[1]:]
                    result = processor.decode(generated_ids, skip_special_tokens=True)

                    exec_end_time = dt.datetime.now()
                    process_time = exec_end_time - exec_start_time
                    process_time = round(process_time.total_seconds(), 4)
                    list_of_words = extract_content(result, model_name)
                    list_of_words_with_conf = [(word, None) for word in list_of_words]

                    predictions_dict[image_key] = {"predictions": list_of_words_with_conf, "time": process_time}

                else:
                    # print(f"Running OCR model {model_name} on dataset {dataset}...")
                    list_of_words_with_conf = []
                    glob_time = 0.0
                    if use_bbox:
                        #print("Bounding box usage is enabled.")
                        # We only treat the images that have bounding boxes in the metadata file
                        for gt_key, coords_list in metadata_file.get(image_key, {}).items():
                            for coords in coords_list:
                                roi = get_resized_roi(image, coords)
                                if roi is None:
                                    raise ValueError(f"Invalid coordinates {coords} for image {image_file}. Could not extract ROI.")

                                exec_start_time = dt.datetime.now()
                                if model_name == "easyocr":
                                    result = reader.readtext(roi, detail=1, paragraph=False)
                                    (bbox, text, prob) = result[0] if result else (None, "", 0)
                                else:
                                    reader = TesseractOCR(lang="eng", psm=8)
                                    result = reader.read_data(roi)  # For pytesseract

                                exec_end_time = dt.datetime.now()
                                process_time = exec_end_time - exec_start_time
                                process_time = round(process_time.total_seconds(), 4)
                                # We add the processing time to the global time for the image
                                glob_time += process_time

                                word, conf = extract_content_from_bbox_results(result, model_name)

                                list_of_words_with_conf.append((word, conf))
                        glob_time = round(glob_time, 4)

                    else:
                        exec_start_time = dt.datetime.now()
                        if model_name == "easyocr":
                            result = reader.readtext(image, detail=1, paragraph=False)
                        else:
                            reader = TesseractOCR(lang="eng", psm=6)
                            result = reader.read_data(image)  # For pytesseract
                        exec_end_time = dt.datetime.now()
                        process_time = exec_end_time - exec_start_time
                        glob_time = round(process_time.total_seconds(), 4)

                        results_list = easyocr_pytesseract_to_word_list(result)

                        list_of_words_with_conf = [(word, conf) for word, conf in results_list]

                    predictions_dict[image_key] = {"predictions": list_of_words_with_conf, "time": glob_time}

            batch_step_time = dt.datetime.now()
            elapsed_time = batch_step_time - last_checkpoint_time
            minutes, seconds = divmod(elapsed_time.total_seconds(), 60)
            seconds, micro_seconds = divmod(seconds, 1)
            last_checkpoint_time = batch_step_time

            save_predictions_checkpoint(checkpoint_path, predictions_dict)
            #print(f"Checkpoint saved after batch [{batch_names[0]} to {batch_names[-1]}]: {len(batch_files)} images in {int(minutes)} minutes and {int(seconds)} seconds.")

            # We had the time took to process the batch.
            batch_number = batch_start // batch_size + 1
            b_key = f"Batch #{batch_number}"
            batch_dict[b_key] = {
                "time": f"{int(minutes)}min_{int(seconds)}s_{int(micro_seconds * 1000)}ms",
                "size": len(batch_files)
            }

        tracker.stop()

        predictions_dict.update({"batch_info": batch_dict})
        # Save the final predictions to a JSON file
        save_predictions_to_json(predictions_dict, dataset, model_name, use_bbox)

## 3.2 Inference

### 3.2.1 Global Inference

In [ ]:
datasets_to_exclude = ["ICDAR15/train", "ICDAR15/test"]
datasets_to_run = [dataset for dataset in avail_datasets if dataset not in datasets_to_exclude]
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"  # Change this to the desired model name
#avail_datasets=['ICDAR03/apanar']

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

benchmark(model_name=model_name,
          use_bbox=False,
          datasets=['icare/train'],
          batch_size=BATCH_SIZE,
          nbr_tokens=NBR_TOKENS
          )

print(f"Pic VRAM allouée: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Pic VRAM réservée: {torch.cuda.max_memory_reserved() / 1024**3:.2f} GB")

### 3.2.2 Unique image inference